In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import time
from pathlib import Path

# Try two common roots
candidates = [Path("../data/processed"), Path("data/processed")]
for c in candidates:
    if (c / "elia_X_train.npy").exists():
        DATA_DIR = c
        print(f"Found data in: {DATA_DIR.resolve()}")
        break
else:
    raise FileNotFoundError("Could not find elia_X_train.npy in ../data/processed or data/processed")

X_train_3d = np.load(DATA_DIR / "elia_X_train.npy").reshape(-1, 24, 1).astype(np.float32)
X_val_3d   = np.load(DATA_DIR / "elia_X_val.npy").reshape(-1, 24, 1).astype(np.float32)
X_test_3d  = np.load(DATA_DIR / "elia_X_test.npy").reshape(-1, 24, 1).astype(np.float32)

print("Loaded:", X_train_3d.shape, X_val_3d.shape, X_test_3d.shape)
device = torch.device("cpu")
print("Device:", device)

X_train_3d = np.load("../data/processed/elia_X_train.npy").reshape(-1, 24, 1).astype(np.float32)
X_val_3d   = np.load("../data/processed/elia_X_val.npy").reshape(-1, 24, 1).astype(np.float32)
X_test_3d  = np.load("../data/processed/elia_X_test.npy").reshape(-1, 24, 1).astype(np.float32)

Found data in: C:\Users\Lenovo\Desktop\EHMAH\CODE\Final Project\capstone\data\processed
Loaded: (33700, 24, 1) (7221, 24, 1) (7222, 24, 1)
Device: cpu


In [2]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size=1, hidden=64, latent=16):
        super().__init__()
        self.encoder_lstm = nn.LSTM(input_size, hidden, batch_first=True)
        self.encoder_fc   = nn.Linear(hidden, latent)
        self.decoder_fc   = nn.Linear(latent, hidden)
        self.decoder_lstm = nn.LSTM(hidden, input_size, batch_first=True)

    def forward(self, x):
        _, (h, _) = self.encoder_lstm(x)
        h = h.squeeze(0)
        z = self.encoder_fc(h)
        h_dec = self.decoder_fc(z).unsqueeze(1).repeat(1, 24, 1)
        out, _ = self.decoder_lstm(h_dec)
        return out, z

model = LSTMAutoencoder(hidden=64, latent=16).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 19,548


In [3]:
batch_size = 256
train_ds = TensorDataset(torch.tensor(X_train_3d))
val_ds   = TensorDataset(torch.tensor(X_val_3d))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

EPOCHS, LR = 30, 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    running = 0.0
    for (batch,) in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, _ = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        running += loss.item() * batch.size(0)
    train_loss = running / len(train_ds)

    model.eval()
    running = 0.0
    with torch.no_grad():
        for (batch,) in val_loader:
            batch = batch.to(device)
            recon, _ = model(batch)
            running += criterion(recon, batch).item() * batch.size(0)
    val_loss = running / len(val_ds)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train {train_loss:.6f} | val {val_loss:.6f} | {time.time()-t0:.1f}s")

Epoch  1/30 | train 0.089730 | val 0.020744 | 23.7s
Epoch  2/30 | train 0.016223 | val 0.017243 | 23.0s
Epoch  3/30 | train 0.015762 | val 0.017114 | 22.5s
Epoch  4/30 | train 0.015648 | val 0.017041 | 22.5s
Epoch  5/30 | train 0.015586 | val 0.017005 | 23.9s
Epoch  6/30 | train 0.015555 | val 0.016975 | 22.9s
Epoch  7/30 | train 0.015534 | val 0.016958 | 24.5s
Epoch  8/30 | train 0.015522 | val 0.016956 | 25.0s
Epoch  9/30 | train 0.015510 | val 0.016933 | 19.8s
Epoch 10/30 | train 0.015494 | val 0.016922 | 22.7s
Epoch 11/30 | train 0.015485 | val 0.016922 | 22.7s
Epoch 12/30 | train 0.015472 | val 0.016905 | 25.1s
Epoch 13/30 | train 0.015463 | val 0.016896 | 24.5s
Epoch 14/30 | train 0.015463 | val 0.016893 | 26.6s
Epoch 15/30 | train 0.015456 | val 0.016889 | 22.7s
Epoch 16/30 | train 0.015454 | val 0.016885 | 23.5s
Epoch 17/30 | train 0.015451 | val 0.016894 | 25.5s
Epoch 18/30 | train 0.015447 | val 0.016885 | 23.7s
Epoch 19/30 | train 0.015449 | val 0.016884 | 32.1s
Epoch 20/30 

In [4]:
def extract_features(X_3d, model, batch_size=512):
    model.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, len(X_3d), batch_size):
            batch = torch.tensor(X_3d[i:i+batch_size]).to(device)
            _, z = model(batch)
            feats.append(z.cpu().numpy())
    return np.vstack(feats)

F_train = extract_features(X_train_3d, model)
F_val   = extract_features(X_val_3d,   model)
F_test  = extract_features(X_test_3d,  model)

np.save("../data/processed/elia_F_train_v2.npy", F_train)
np.save("../data/processed/elia_F_val_v2.npy",   F_val)
np.save("../data/processed/elia_F_test_v2.npy",  F_test)
torch.save(model.state_dict(), "../models/lstm_autoencoder_v2.pt")

print("F_train:", F_train.shape)
print("Saved v2 features and model.")

F_train: (33700, 16)
Saved v2 features and model.


In [5]:
from sklearn.ensemble import RandomForestRegressor

y_train = np.load("../data/processed/elia_y_train.npy")
y_test  = np.load("../data/processed/elia_y_test.npy")

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))) * 100

rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
rf.fit(F_train, y_train)
pred = rf.predict(F_test)

print(f"Latent v2 (16 features): MAPE {mape(y_test, pred):.4f}%")
print("Compare to raw (24): 1.5023%")

Latent v2 (16 features): MAPE 3.5932%
Compare to raw (24): 1.5023%
